# Phase 1 — Data understanding

Uses the same service layer as `scripts/phase1_overlap.py`: entity counts, raw-material usage, repeated materials (optional thresholds), and supplier fragmentation.

**Note:** In the challenge SQLite, each raw material row appears in BOMs for finished goods from **one company only** (`n_companies` is always 1). Cross-company overlap must be inferred in Phase 2 via name normalization / clustering, not raw `Product.Id`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "raw" / "db.sqlite").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from agnes.config.settings import Settings
from agnes.data.db_loader import get_engine
from agnes.data import queries
from agnes.services.overlap import build_phase1_report, compute_supplier_fragmentation

settings = Settings(db_path=ROOT / "data" / "raw" / "db.sqlite")
engine = get_engine(settings)

In [ ]:
report = build_phase1_report(engine)
print(report.entity_counts.model_dump())
usage = queries.raw_material_usage(engine)
display(usage.head())
print("n_companies max:", usage["n_companies"].max())

In [ ]:
frag = compute_supplier_fragmentation(engine, min_suppliers=2)
top = frag[:8]
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh([x.sku[:40] + "…" if len(x.sku) > 40 else x.sku for x in top], [x.supplier_count for x in top])
ax.set_xlabel("Distinct suppliers")
ax.set_title("Top supplier fan-out (raw materials with ≥2 suppliers)")
plt.tight_layout()
plt.show()